# Coding Session #7

---

## Today's session

Today's session is structured to teach more complex tasks. We will explore:

- For loops (a bit more advanced)
- apply function
- seaboarn and facetting

### 0.1 Environment preparation

We begin by loading the libraries we’ll need. In Python, libraries are like toolkits: they extend the language with specialized functions.

- **pandas (pd)** is our main tool for working with tabular data. It introduces the DataFrame, which lets us manipulate datasets in a way that feels natural if you’ve used Excel or R.
- **NumPy (np)** provides the numerical backbone. It gives us arrays and fast mathematical functions, which pandas actually uses under the hood.
- **Seaborn (sns)** builds on top of Matplotlib to create a vast range of plots and visualization.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns


### 0.2 Reading Dataset


The **Database on Ideology, Money in Politics, and Elections (DIME), Version 4.0** provides a comprehensive record of campaign finance activity in the United States between **1979 and 2024**. It contains more than **850 million itemized contributions** from individuals and organizations to candidates, parties, and political committees at the federal, state, and local levels (Bonica, 2024).

DIME is organized into two primary components:

* **Candidate/Recipient Database** – includes information on candidates and committees, such as names, party affiliation, state, office sought, district, gender, incumbency status, fundraising totals, election outcomes, and multiple measures of ideology.
* **Contributor Database** – contains records on donors, both individuals and organizations, with demographic details, contribution histories, and ideological scores.

For this coding session, we use a **subset of the candidate/recipient database** covering the **2020 and 2024 federal elections** with some small modifications.

You can access the complete dataset here: [DIME 4.0 – Stanford University Libraries](http://data.stanford.edu/dime).

**Reference:**
Bonica, Adam. 2024. *Database on Ideology, Money in Politics, and Elections: Public version 4.0* [Computer file]. Stanford, CA: Stanford University Libraries. Available at: [http://data.stanford.edu/dime](http://data.stanford.edu/dime).

#### Codebook


| Variable                | Description |
|--------------------------|-------------|
| **lname**               | Last name of the candidate/recipient. |
| **fname**               | First name of the candidate/recipient. |
| **party**               | Party of candidate/recipient (100 = Democrat, 200 = Republican, 328 = Independent). |
| **state**               | Two-letter state abbreviation. |
| **seat**                | Office sought (e.g., `federal:house`, `federal:senate`, `federal:president`, `state:governor`, etc.). |
| **district**            | District code: two-letter state code followed by congressional district number. For Senate candidates, “S” plus the year of the seat’s election. |
| **ico.status**          | Incumbency status (`I` = Incumbent, `C` = Challenger, `O` = Open Seat Candidate, blank = not up for election). |
| **cand.gender**         | Candidate gender coding (based on Census first-name ratios and gendered titles). |
| **recipient.cfscore.dyn** | Period-specific estimates of ideology. Candidate/recipient scores are re-estimated each cycle while holding contributor scores constant. |
| **contributor.cfscore** | Estimated ideology of the candidate/recipient based on their **personal donations** to other candidates/recipients. |
| **num.givers.total**    | Number of distinct donors that gave to the candidate/recipient over their career. |
| **total.receipts**      | Sum total of contributions raised during an election cycle. |
| **total.disbursements**      | Sum total of disbursements spent during an election cycle. |
| **prim.vote.pct**       | FEC-reported vote share (%) in the primary election (federal congressional candidates only). |
| **pwinner**             | Primary election outcome (`W` = won; `L` = lost). Federal congressional candidates only. |
| **gen.vote.pct**        | FEC-reported vote share (%) in the general election. |
| **gwinner**             | General election outcome (`W` = won; `L` = lost). Federal candidates coded from FEC, state candidates from NIMSP. |



In [ ]:
#Load "dime_candidates_2020_2024.csv" from remote repository
df_bonica = pd.read_csv("https://raw.githubusercontent.com/albertostefanelli/data_science_campaigns/refs/heads/master/coding_sessions/session_06/data/dime_candidates_2020_2024_clean.csv")

# Quick check
df_bonica.head()
df_bonica.dtypes

## 1. It is still unclear if ideologically moderate candidates are better fundraisers than non-moderates?

In our previous cut, moderates appeared to raise less overall, but that gap lost precision under a permutation test. Several mechanisms could be at play:

- Measurement of "moderate": maybe our cut-off for definign who is moderate and who is not is incorrect.
- Incumbents, safe-seat candidates, and high-quality challengers systematically differ in both moderation and donor access


### Let's take a quick look at the function we wrote last time

- **Moderates vs non-moderates**. Take the ideological score and classify candidates as moderate or non-moderate based on a chosen range around the center (e.g., ±0.5).
- **Mean difference** compute the difference in the outcome means between moderates and non-moderates, This will be: $\text{Mean Difference} = \overline{Y}_{\text{moderate}=1} - \overline{Y}_{\text{moderate}=0}$
- **Permutation test:** Randomly shuffle the “moderate” labels many times, recalculate the mean difference for each shuffle, and compare the observed value to this simulated distribution.

In [ ]:
def make_moderate_indicator(df, ideology_col, tau=0.5, colname="moderate"):
    """Add a binary column: 1 if |ideology| <= tau, else 0."""
    df[colname] = (df[ideology_col].abs() <= tau).astype(int)
    return df

In [ ]:
def mean_diff(df, group_col, outcome_col):
    """
    Compute the difference in mean outcomes between two groups.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataset containing both the grouping and outcome variables.
    group_col : str
        The name of the binary treatment column (1 = focal group, 0 = comparison group).
    outcome_col : str
        The name of the outcome variable (numeric).

    Returns
    -------
    float
        The difference in mean outcome between the two groups.
        (mean_outcome_focal - mean_outcome_comparison)
    """

    # 1. Keep only the relevant columns and remove missing data
    data = df[[group_col, outcome_col]].dropna()

    # 2. Separate the groups
    focal_group = data.loc[data[group_col] == 1, outcome_col]
    comparison_group = data.loc[data[group_col] == 0, outcome_col]

    # 3. Compute mean outcomes
    mean_focal = focal_group.mean()
    mean_comparison = comparison_group.mean()

    # 4. Compute the difference in means
    diff = mean_focal - mean_comparison

    return diff


In [ ]:
# Let's build a permutation test function

def permutation_test(df, group_col, outcome_col, n_permutations=1000):
    """
    Simple permutation test for the difference in means.

    Steps:
    1. Compute the observed difference in means using mean_diff().
    2. Shuffle the focal labels many times and recompute the difference.
    3. Compare the observed difference to the shuffled differences to get a p-value.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataset containing both the grouping (group_col) and outcome (outcome_col).
    group_col : str
        The column with a binary grouping variable (e.g., 1 = moderates, 0 = extremists).
    outcome_col : str
        The numeric outcome variable (e.g., total.receipts).
    n_permutations : int
        Number of random shuffles to perform (default = 1000).
    """

    # --- Step 1. Compute the observed difference in means ---
    obs_diff = mean_diff(df, group_col, outcome_col)

    # --- Step 2. Prepare an array to store results ---
    perm_diffs = np.empty(n_permutations)

    # --- Step 3. Run the permutation test loop ---
    for i in range(n_permutations):
        # Randomly shuffle the labels (moderate vs. extreme)
        shuffled = np.random.permutation(df[group_col])

        # Create a temporary copy with shuffled labels
        df_shuffled = df.copy()
        df_shuffled[group_col] = shuffled

        # Compute the difference in means for this shuffled data
        perm_diffs[i] = mean_diff(df_shuffled, group_col, outcome_col)


    return obs_diff, perm_diffs



## 2. Exploring different thresholds for “Moderate”

So far, we’ve treated “moderate” as a fixed category — someone whose ideology score lies within ±0.5 of the center. But this cutoff is arbitrary. What happens if we shift that threshold? Do our conclusions about how moderates differ from non-moderates depend on where we draw the line?

To find out, we’ll systematically vary the threshold $τ$ that defines who counts as moderate. By doing this, we can see how sensitive our results are to different operationalizations of moderation.

In [ ]:
taus = np.arange(0.1, 2.8, 0.1)

results = []

for t in taus:
    temp_df = df_bonica.copy()

    temp_df = make_moderate_indicator(temp_df, ideology_col="recipient.cfscore.dyn", tau=t)
    n_mod = temp_df["moderate"].value_counts().loc[1]
    temp_df[["total.receipts"]] = temp_df[["total.receipts"]]/1000
    diff = mean_diff(temp_df, group_col="moderate", outcome_col="total.receipts")

    results.append({"tau": t, "mean_diff": diff, "n":n_mod})

results_df = pd.DataFrame(results)
print(results_df)


Ok, the story seems a bit more complicated than what we thought. Let's plot our original variable and annotate it based on this results

In [ ]:
df_bonica_plot = df_bonica[['recipient.cfscore.dyn']].dropna().copy()
df_bonica_plot.rename(columns={'recipient.cfscore.dyn': 'ideology'}, inplace=True)

In [ ]:
hist_ideo = sns.histplot(df_bonica_plot['ideology'],
             bins=40,
             kde=True, color='skyblue')

hist_ideo.axvspan(-0.8, 0.8, color='purple', alpha=0.2)
hist_ideo.axvspan(0.8, 1.7, color='lightgreen', alpha=0.2)
hist_ideo.axvspan(-0.8, -1.7, color='lightgreen', alpha=0.2)

hist_ideo.axvspan(1.7, 2.0, color='red', alpha=0.2)
hist_ideo.axvspan(-1.7, -2, color='red', alpha=0.2)
hist_ideo.axvspan(2, 2.8, color='lightgreen', alpha=0.2)
hist_ideo.axvspan(-2, -2.8, color='lightgreen', alpha=0.2)

hist_ideo.set(
    title="Distribution of Candidate Ideology (CFscore)",
    xlabel="Ideological Score (Negative = Liberal, Positive = Conservative)",
    ylabel="Number of Candidates"
)

What we can make out of this:

- True centrists, those located near the middle of the ideological distribution, receive substantially fewer donations than their more clearly ideological counterparts.
- Around τ = 0.9, the mean difference turns positive, indicating that these “soft partisans” — candidates who lean slightly liberal or conservative — now out-raise more extreme candidates.
- As τ increases beyond 0.9, the category begins to include the ideological mainstream — roughly center-left liberals and center-right conservatives. Together, these groups attract significantly more donations than candidates at the extremes but less and less compared to soft partisans.
- Candidates with ideological scores above +2 (on the far-right end of the CFscore scale, and similarly at the far-left tail) tend to raise less money than the rest of the ideological spectrum, reflecting limited donor appeal outside the mainstream.

## 3. Categorizing candidates by ideological profile

Now that we’ve examined how τ shapes fundraising differences, we can translate those findings into substantive ideological categories based on each candidate’s CFscore (which ranges from about –2 to +2.8).

This allows us to move from a continuous ideological scale to meaningful analytical groups that mirror what we observed in the data:

- True centrists: very close to 0, politically neutral, least funded
- Soft partisans: slightly ideological moderates with growing donor appeal
- Mainstream: center-left and center-right, peak donor support
- Extremes: far-left and far-right, narrower appeal  


In [ ]:
def categorize_ideology(score):
    if -0.5 <= score <= 0.5:
        return "True centrist"
    elif (-0.9 <= score < -0.5) or (0.5 < score <= 0.9):
        return "Soft partisan"
    elif (-1.7 <= score < -0.9) or (0.9 < score <= 1.7):
        return "Mainstream"
    else:
        return "Extreme"

To assign each candidate to one of these categories, we use the .apply() method in pandas.

.apply() runs a function on every element of a Series — in this case, the ideology column — and stores the returned value in a new column.

So for every candidate:

- Their ideology value (e.g., +1.2) is passed to categorize_ideology().
- The function checks which range it falls into.
- The corresponding label (e.g., "Mainstream") is returned and stored in a new column.

In [ ]:
df_bonica["ideology_c"] = df_bonica["recipient.cfscore.dyn"].apply(categorize_ideology)

In [ ]:
df_bonica["ideology_c"].value_counts().sort_index()


In [ ]:
# express donations in milions
df_bonica["total.receipts_m"] = df_bonica["total.receipts"]/1000000

mean_donations = (
    df_bonica
    .groupby("ideology_c")["total.receipts_m"]
    .mean()
)

mean_donations

## 4. Permutation test True Centrists vs. Other Ideological Groups [Home Work]

To test whether True Centrists raise significantly different amounts of money than each of the other ideological groups using a permutation test within a loop.

In [ ]:
# List of all ideological categories
categories = ["Soft partisan", "Mainstream", "Extreme"]

results = []

for cat in categories:
    # Subset to True Centrists and the comparison group
    subset = df_bonica[
        df_bonica["ideology_c"].isin(["True centrist", cat])
    ].copy()

    # Binary variable: 1 = True Centrist, 0 = comparison category
    subset["is_true_centrist"] = (subset["ideology_c"] == "True centrist").astype(int)

    # Run permutation test
    obs_diffs, perm_diffs = permutation_test(subset, "is_true_centrist", "total.receipts", 10000)

    results.append({
        "Comparison": f"True centrist vs {cat}",
        "obs_diffs": obs_diffs,
        "perm_diffs": perm_diffs

        })

# Display results as a DataFrame
perm_results = pd.DataFrame(results)
display(perm_results)


In [ ]:
df_long = perm_results.explode("perm_diffs").reset_index(drop=True)
df_long

In [ ]:
print(df_long.columns)

In [ ]:
import matplotlib.pyplot as plt   # ← this line is required!
# facet grid with histograms
g = sns.FacetGrid(df_long, col="Comparison", col_wrap=3, sharex=False, sharey=False)
g.map_dataframe(sns.histplot, x="perm_diffs", bins=30)

# Add vertical line for each facet
g.map_dataframe(
    lambda data, **_: plt.axvline(data["obs_diffs"].iloc[0], color="red", linestyle="--", linewidth=2)
)


# Congratulations!

You are done with the coding session. Questions or suggestions? Email Alberto at alberto.stefanelli@yale.edu

In [ ]:
# Install requirements
!apt-get -qq update
!apt-get install -y pandoc texlive-xetex texlive-fonts-recommended texlive-plain-generic

from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive')

# Ask for the notebook name
notebook_name = input(
    "Enter your notebook’s exact file name,\n"
    "exactly as shown in the top-left corner of the Colab page (next to the two yellow circle icons): "
)

# Build paths
input_path = f"/content/drive/MyDrive/Colab Notebooks/{notebook_name}"
output_path = input_path.replace(".ipynb", ".pdf")

# Convert to PDF
!jupyter nbconvert --to pdf "{input_path}"

# Download the PDF
files.download(output_path)